# 1. Install Dependencies

In [2]:
# Install all dependencies in one go
%%capture
!pip install langchain langchain-classic langchain-community sentence-transformers chromadb transformers accelerate textstat ragas datasets xlsxwriter openpyxl ollama langchain-ollama langchain-google-genai

In [4]:
#mount gdrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# All Consolidated Imports
import subprocess
import time
import requests
import os
import ast
import pandas as pd
import textstat
from datasets import Dataset

# For Embeddings and Vector DB
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# For LLM and RAG Chain
import ollama
from langchain_ollama.llms import OllamaLLM
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

# For Ragas Evaluation
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from ragas import evaluate
from ragas.metrics import faithfulness, answer_correctness, context_precision
from ragas.run_config import RunConfig


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2608/3198652380.py:29: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_correctness, context_precision
/tmp/ipykernel_2608/3198652380.py:29: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collectio

# 2. Load Embeddings and Vector DB

In [6]:
# @title Database Configuration
DB_PATH = "/content/drive/MyDrive/Colab Notebooks/GenAI_Grp/ada_vdb" # @param {type:"string"}
COLLECTION_NAME = "ADA_Diabetes_db_20260312" # @param {type:"string"}
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5" # @param {type:"string"}

In [7]:
# 1. Initialize the raw client using params
client = chromadb.Client(
    Settings(
        persist_directory=DB_PATH,
        is_persistent=True
        )
)

# 2. List all collections in that folder to see what is actually saved
print("Collections found in this directory:")
for c in client.list_collections():
    print(f"- {c.name}")

# 3. Try to get the specific collection and count the documents
try:
    collection = client.get_collection(name=COLLECTION_NAME)
    doc_count = collection.count()
    print(f"\nSuccess! The '{COLLECTION_NAME}' collection exists and contains {doc_count} documents.")

    if doc_count > 0:
        # Let's peek at the first document to make sure it's readable
        peek_data = collection.peek(1)
        print("\nSnippet of the first document stored:")
        print(peek_data['documents'][0][:150])

except ValueError:
    print(f"\nError: The collection '{COLLECTION_NAME}' does not exist in this folder.")

Collections found in this directory:
- ADA_Diabetes_db_20260312

Success! The 'ADA_Diabetes_db_20260312' collection exists and contains 3022 documents.

Snippet of the first document stored:
2. DIAGNOSIS AND CLASSIFICATION OF DIABETES 2. Diagnosis and Classification of Diabetes: Standards of Care in Diabetes—2026 Diabetes Care 2026;49(Supp


In [8]:
# 1. Wrap the embedding model
langchain_embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME
)

# 2. Create the LangChain VectorStore
vector_db = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=langchain_embeddings
)

# 3. Define the retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 3})


# 4. Initialize the SentenceTransformer for direct query encoding
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

/tmp/ipykernel_2608/3688915987.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  langchain_embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_2608/3688915987.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Test search (to retrieve chunks)
query = "What are symptoms of diabetic neuropathy?"

# Encode the query
query_embedding = embedding_model.encode(query)

# Use the 'collection' object obtained from chromadb.Client
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=['documents', 'metadatas'] # Include documents and metadatas in the results
)

print("Query results:")
for i, doc_content in enumerate(results['documents'][0]):
    print(f"\n--- Document {i+1} ---")
    print(f"Metadata: {results['metadatas'][0][i]}")
    print(f"Content: {doc_content[:200]}...") # Print a snippet of the content

Query results:

--- Document 1 ---
Metadata: {'chunk_index': 1323, 'topic': 'Diabetes'}
Content:  all individuals with type 2 diabetes should be assessed annually for autonomic neuropathy (63). The symptoms and signs of autonomic neuropathy should be elicited carefully during the history and phys...

--- Document 2 ---
Metadata: {'chunk_index': 2906, 'topic': 'Diabetes'}
Content: penia, inflammation, and insulin resistance (131). Diabetic peripheral neuropathy (DPN) is a common complication of both type 1 and 2 diabetes and may cause impaired postural balance and gait kinemati...

--- Document 3 ---
Metadata: {'topic': 'Diabetes', 'chunk_index': 1315}
Content: ous complication, as it can exacerbate cardiovascular disease and contribute to heart failure and sudden cardiac death (62). Screening Diagnosis Diabetic Peripheral Neuropathy Individuals who have had...


# 3. Initialise the LLM


*   1) Mistral 7B https://huggingface.co/mistralai/Mistral-7B-v0.1
*   2) Llama 8B https://huggingface.co/meta-llama/Meta-Llama-3-8B
*   3) Qwen 3.5 9B https://huggingface.co/Qwen/Qwen3.5-9B



In [10]:
#setup Ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (552 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently i

In [11]:
# connect to server
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print('Waiting for Ollama server to start...')
for i in range(30):
    try:
        r = requests.get('http://localhost:11434')
        if r.status_code == 200:
            print(f'✅ Ollama server is ready! (took ~{i+1}s)')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('❌ Ollama server did not start in time. Try re-running this cell.')

Waiting for Ollama server to start...
✅ Ollama server is ready! (took ~2s)


### Pulling Mistral 7B, Llama 8B, and Qwen 3.5 9B with Ollama

In [12]:
# Pull Mistral 7B
!ollama pull mistral:7b

# Pull Llama 8B (Llama 3 8B)
!ollama pull llama3:8b

# Pull Qwen 3.5 9B
!ollama pull qwen3.5:9b

!ollama list




NAME          ID              SIZE      MODIFIED               
qwen3.5:9b    6488c96fa5fa    6.6 GB    Less than a second ago    
llama3:8b     365c0bd3c000    4.7 GB    3 minutes ago             
mistral:7b    6577803aa9a0    4.4 GB    4 minutes ago             


In [13]:
# Initialize Ollama with phi4 or llama3.2, and define the prompt for the RAG chain.

# 1. Initialize Ollama

# llm = OllamaLLM(model="llama3.2")
llm_mistral = OllamaLLM(model="mistral:7b",
                        temperature=0.0,
                        seed=42)
llm_llama = OllamaLLM(model="llama3:8b",
                        temperature=0.0,
                        seed=42)
llm_qwen = OllamaLLM(model="qwen3.5:9b",
                        temperature=0.0,
                        seed=42)

llm = llm_mistral # @param {type:"string"}

# 2. Define the explicit prompt
prompt_template = """
You are a helpful medical assistant. Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. Do not make up information.

Context:
{context}

Question:
{input}

Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)

# 3. Create the chains (using your existing 'retriever' from earlier)
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

In [14]:
# 4. Test your baseline!
query = "What are symptoms of diabetic neuropathy?"
response = rag_chain.invoke({"input": query})

print(response['answer'])

 Symptoms of diabetic neuropathy include resting tachycardia, orthostatic hypotension, gastroparesis, constipation or diarrhea, fecal incontinence, erectile dysfunction, neurogenic bladder, sudomotor dysfunction with either increased or decreased sweating, inflammation, insulin resistance, impaired postural balance, and gait kinematics issues. Early symptoms may also include numbness, tingling, burning pain, or sharp pains in the feet or hands.


In [15]:
# 5. Review the 'Source' context to verify grounding
print("\n--- RETRIEVED CONTEXT ---")
for i, doc in enumerate(response['context']):
    print(f"Source {i+1}: {doc.page_content[:200]}...")


--- RETRIEVED CONTEXT ---
Source 1:  all individuals with type 2 diabetes should be assessed annually for autonomic neuropathy (63). The symptoms and signs of autonomic neuropathy should be elicited carefully during the history and phys...
Source 2: penia, inflammation, and insulin resistance (131). Diabetic peripheral neuropathy (DPN) is a common complication of both type 1 and 2 diabetes and may cause impaired postural balance and gait kinemati...
Source 3: ous complication, as it can exacerbate cardiovascular disease and contribute to heart failure and sudden cardiac death (62). Screening Diagnosis Diabetic Peripheral Neuropathy Individuals who have had...


#Eval

## Load Eval Dataset & Define generation loop

In [16]:
# pd.set_option for display
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [17]:
#Load dataset
eval_path= "/content/drive/MyDrive/Colab Notebooks/GenAI_Grp"
eval_df = pd.read_csv(os.path.join(eval_path, "ADA_diabetes_QA_evaluationset_20260314.csv"))
print(f"Eval_df length: {len(eval_df)}")

Eval_df length: 97


In [18]:
#check loaded df
test_df = eval_df.head(1)
test_df

,Unnamed: 0,index,Persona,Qn type,Context,Question,Golden Answer,Reference
0,0,1,Professional,Simple Qn,Glycemic Assessment,What are the primary tools used to assess glycemic status in a patient with diabetes?,"Glycemic status is assessed using A1C measurement, BGM by capillary finger-stick devices, and CGM metrics including TIR, TBR, TAR, GMI, coefficient of variation, and mean glucose.","Chapter 6: ""Glycemic status is assessed by A1C measurement, blood glucose monitoring (BGM) by capillary (finger-stick) devices, and different continuous glucose monitoring (CGM) metrics such as time in range (TIR), time below range (TBR), time above range (TAR), glucose management indicator (GMI), coefficient of variation, and mean glucose."""


In [1]:
def run_generation_loop(df, rag_chain):
    results = []

    print(f"Starting Generation for ({len(df)} questions)...")

    for index, row in df.iterrows():
        # Convert the current row to a dictionary to include all its columns
        row_data = row.to_dict()
        question = row_data['Question']

        # Start timer for latency tracking
        start_time = time.time()

        # Run Vanilla RAG (Assuming your chain is called `rag_chain`)
        response = rag_chain.invoke({"input": question})

        latency = time.time() - start_time

        # Extract the retrieved chunks
        retrieved_contexts = [doc.page_content for doc in response['context']]

        # Add generated content to the row dictionary
        row_data.update({
            "Generated Answer": response['answer'],
            "Retrieved Contexts": retrieved_contexts,
            "Latency Seconds": latency,
        })

        results.append(row_data)

        print(f"Processed {index + 1}/{len(df)} in {latency:.2f}s")

    return results

##mode: Vanilla RAG (ada only)

### Generate Responses

In [19]:
# Test generation
test_df = eval_df.head(1)
test_results_df = run_generation_loop(test_df, rag_chain)
test_results_df

Starting Generation for (1 questions)...
Processed 1/1 in 4.96s


[{'Unnamed: 0': 0,
  'index': 1,
  'Persona': 'Professional',
  'Qn type': 'Simple Qn',
  'Context': 'Glycemic Assessment',
  'Question': 'What are the primary tools used to assess glycemic status in a patient with diabetes?',
  'Golden Answer': 'Glycemic status is assessed using A1C measurement, BGM by capillary finger-stick devices, and CGM metrics including TIR, TBR, TAR, GMI, coefficient of variation, and mean glucose.',
  'Reference': 'Chapter 6: "Glycemic status is assessed by A1C measurement, blood glucose monitoring (BGM) by capillary (finger-stick) devices, and different continuous glucose monitoring (CGM) metrics such as time in range (TIR), time below range (TBR), time above range (TAR), glucose management indicator (GMI), coefficient of variation, and mean glucose."',
  'Generated Answer': " The primary tools used to assess glycemic status in a patient with diabetes include self-monitoring of blood glucose (BGM), Continuous Glucose Monitoring (CGM), and the A1C test. Howeve

In [20]:
# Generate Results
eval_results_df = run_generation_loop(eval_df, rag_chain)

Starting Generation for (97 questions)...
Processed 1/97 in 4.28s
Processed 2/97 in 3.41s
Processed 3/97 in 1.84s
Processed 4/97 in 4.92s
Processed 5/97 in 5.75s
Processed 6/97 in 4.33s
Processed 7/97 in 5.34s
Processed 8/97 in 10.10s
Processed 9/97 in 1.63s
Processed 10/97 in 5.90s
Processed 11/97 in 4.67s
Processed 12/97 in 1.94s
Processed 13/97 in 6.78s
Processed 14/97 in 7.29s
Processed 15/97 in 6.58s
Processed 16/97 in 5.66s
Processed 17/97 in 9.16s
Processed 18/97 in 9.65s
Processed 19/97 in 1.35s
Processed 20/97 in 3.27s
Processed 21/97 in 1.01s
Processed 22/97 in 2.90s
Processed 23/97 in 8.70s
Processed 24/97 in 2.08s
Processed 25/97 in 15.30s
Processed 26/97 in 10.28s
Processed 27/97 in 3.50s
Processed 28/97 in 8.14s
Processed 29/97 in 4.62s
Processed 30/97 in 6.30s
Processed 31/97 in 3.27s
Processed 32/97 in 3.69s
Processed 33/97 in 2.79s
Processed 34/97 in 5.81s
Processed 35/97 in 3.73s
Processed 36/97 in 5.22s
Processed 37/97 in 7.66s
Processed 38/97 in 6.25s
Processed 39/9

In [24]:
# Save results

# Define the path to the Excel file
excel_file_path = os.path.join(eval_path, "Eval Results.xlsx")

# Read existing sheets to preserve them
# Use ExcelFile to get all sheet names dynamically
with pd.ExcelFile(excel_file_path) as xls:
    existing_sheets = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}

# Add the new results to the dictionary of sheets
tab_name = 'ada_mistral_new' # @param {type:"string"}
existing_sheets[tab_name] = pd.DataFrame(eval_results_df)

# Write all sheets back to the Excel file using openpyxl engine
with pd.ExcelWriter(excel_file_path, engine='openpyxl') as writer:
    for sheet_name, df_to_write in existing_sheets.items():
        df_to_write.to_excel(writer, sheet_name=sheet_name, index=False)

print("Generation Complete and results saved to 'Eval_Set.xlsx' in a new tab!")

Generation Complete and results saved to 'Eval_Set.xlsx' in a new tab!
